In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import tensorflow as tf

In [2]:
from keras.layers import Input,Dense,Flatten
from keras.models import Model
from keras.optimizers import Adam

from keras.preprocessing import image

import numpy as np
import glob
from keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
from datetime import datetime
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input


In [3]:
# Define the image size
IMAGE_SIZE = [224, 224, 3]

# Load the model
resnet50 = ResNet50(include_top=False, input_shape=IMAGE_SIZE, weights='imagenet')

# Visualize the model summary
#resnet50.summary()


94765736/94765736 [==============================] - 1s 0us/step


In [4]:
for layer in resnet50.layers:
    layer.trainable = False

In [ ]:
x = Flatten()(resnet50.output)

# Created a new layer as output
prediction = Dense(7, activation='softmax')(x)

# Join it with the model
model = Model(inputs=resnet50.input, outputs=prediction)

In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 230, 230, 3)          0         ['input_1[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 112, 112, 64)         9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1_conv[0][0]']          
 on)                                                                                          

In [ ]:
from tensorflow.keras.optimizers import RMSprop

# Define RMSprop optimizer with a smaller learning rate
rmsprop = RMSprop(learning_rate=0.001)  # Adjust the learning rate as needed

# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])


In [5]:
train_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Train'
test_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test'

In [9]:
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


# Train data
train_set = train_datagen.flow_from_directory(train_path,
                                              target_size=(224, 224),
                                              batch_size=32,
                                              class_mode='categorical')

# Test data
test_set = test_datagen.flow_from_directory(test_path,
                                            target_size=(224, 224),
                                            batch_size=32,
                                            class_mode='categorical')

Found 6300 images belonging to 7 classes.
Found 1580 images belonging to 7 classes.


In [ ]:
output_layer = model.layers[-1]  # Assuming the output layer is the last layer in the model
num_classes = output_layer.output_shape[-1]  # Number of units in the output layer

print("Number of classes in the output layer:", num_classes)


Number of classes in the output layer: 7


In [ ]:
import os
os.getcwd()


'/content'

In [6]:
import os
from sklearn.utils import class_weight
import numpy as np

# Path to your dataset
dataset_path = train_path

# Function to count images in each class directory
def count_images_in_classes(dataset_path):
    labels = []
    class_names = os.listdir(dataset_path)
    for class_index, class_name in enumerate(class_names):
        class_dir = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_dir):
            num_images = len([img_name for img_name in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, img_name))])
            labels.extend([class_index] * num_images)
    return np.array(labels), class_names

# Count images and get labels
y_train, class_names = count_images_in_classes(dataset_path)

# Compute class weights
class_weights = class_weight.compute_class_weight('balanced',
                                                  classes=np.unique(y_train),
                                                  y=y_train)

class_weights_dict = dict(zip(np.unique(y_train), class_weights))

# Print computed class weights
print("Computed class weights:")
print(class_weights_dict)


Computed class weights:
{0: 0.9656652360515021, 1: 0.5067567567567568, 2: 2.132701421800948, 3: 1.3824884792626728, 4: 1.3062409288824384, 5: 1.4218009478672986, 6: 0.7518796992481203}


In [7]:
from keras.models import load_model
model = load_model("/content/drive/MyDrive/Models/resnet50.keras")


In [12]:
# Define the file name for the model checkpoint
checkpoint_filepath = '/content/drive/MyDrive/Models/resnet50v2.keras'

# Define the ModelCheckpoint callback
checkpoint = ModelCheckpoint(filepath=checkpoint_filepath, verbose=1, save_best_only=True)

# Combine all callbacks
callbacks = [checkpoint]

# Start timing
start = datetime.now()

# Train the model
model_history = model.fit(train_set,
                          validation_data=test_set,
                          epochs=5,
                          steps_per_epoch=197,
                          validation_steps=50,
                          callbacks=callbacks,
                          verbose=1,
                          class_weight=class_weights_dict)  # Corrected argument name

# Calculate duration
duration = datetime.now() - start

print('Total elapsed time:', duration)


Epoch 1/5
  3/197 [..............................] - ETA: 1:16:15 - loss: 0.0112 - accuracy: 1.0000

KeyboardInterrupt: 